# 01 — Getting started with PySpark

This notebook covers Phase 1 of the roadmap: creating a SparkSession, loading data, and basic DataFrame operations.

Before running: make sure you've generated the sample data with `python scripts/generate_data.py` from the project root.

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# local[*] = run Spark on this machine, using all CPU cores
spark = (
    SparkSession.builder
    .appName("getting-started")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
# While a job runs, the Spark UI is at http://localhost:4040

Spark version: 4.1.2


## Load the sample data

Spark is *lazy*: `spark.read.csv` doesn't actually scan the whole file — work only happens when you call an **action** like `show()` or `count()`.

In [22]:
DATA_DIR = "../data/generated"

orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA_DIR}/products.csv", header=True, inferSchema=True)

orders.printSchema()
orders.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- order_total: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+----------+--------+--------+-----------+----------+
|order_id|customer_id|product_id|quantity|discount|order_total|order_date|
+--------+-----------+----------+--------+--------+-----------+----------+
|       1|        355|        10|       4|     0.1|     1620.0|2023-11-27|
|       2|        286|         4|       3|     0.0|      630.0|2024-07-21|
|       3|        617|         5|       1|    0.05|     284.99|2024-10-05|
|       4|       1118|         3|       4|    0.05|      300.2|2025-05-03|
|       5|        197|         3|       5|     0.1|      355.5|2023-11-17|
+--------+-----------+----------+--------+--------+-----------+----------+
only showing top 5 rows


## Basic operations

`select`, `filter`, `withColumn`, and `orderBy` are **transformations** — they build up a plan without executing it. `show()` triggers execution.

In [23]:
# Orders over $500, biggest first
big_orders = (
    orders
    .filter(F.col("order_total") > 500)
    .select("order_id", "customer_id", "order_total", "order_date")
    .orderBy(F.col("order_total").desc())
)

big_orders.show(10)
print(f"{big_orders.count()} orders over $500")

+--------+-----------+-----------+----------+
|order_id|customer_id|order_total|order_date|
+--------+-----------+-----------+----------+
|     881|        311|    4499.95|2025-02-19|
|    1557|       1463|    4499.95|2025-05-17|
|     935|        123|    4499.95|2024-06-17|
|     317|        732|    4499.95|2023-09-12|
|    1024|        386|    4499.95|2024-11-11|
|     444|        602|    4499.95|2023-01-20|
|    1126|        761|    4499.95|2023-06-03|
|     804|       1467|    4499.95|2024-05-22|
|    1177|       1878|    4499.95|2024-02-06|
|    1203|       1416|    4499.95|2024-06-03|
+--------+-----------+-----------+----------+
only showing top 10 rows
18394 orders over $500


In [ ]:
# Add a derived column, then aggregate: orders per year
orders_with_year = orders.withColumn("year", F.year("order_date"))

orders_with_year.groupBy("year").count().orderBy("year").show()

## A taste of joins (Phase 2 preview)

Join orders to products and compute revenue by category.

In [ ]:
revenue_by_category = (
    orders
    .join(products, on="product_id", how="inner")
    .groupBy("category")
    .agg(
        F.round(F.sum("order_total"), 2).alias("revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("revenue").desc())
)

revenue_by_category.show()

## Your turn

Try these before moving to Phase 2 of the roadmap (see README):

1. How many distinct customers placed at least one order?
2. What's the average `order_total` per `discount` level?
3. Find the 5 most recent orders (`orderBy` on `order_date`).
4. Some orders reference customers that don't exist in `customers.csv` — how many? (Hint: `join` with `how="left_anti"`.)

When you're done, stop the session with the cell below.

In [24]:
## 1. How many distinct customers placed at least one order?
distinct_customers = (
    orders.join(customers, on="customer_id", how="inner")
    .select("customer_id")
    .distinct()
)

print(distinct_customers.count())




2000


In [18]:
## 2. What's the average order_total per discount level?
average_total_by_discount_level = (
    orders.groupby(F.col("discount")).avg("order_total")
)

average_total_by_discount_level.show()

+--------+-----------------+
|discount| avg(order_total)|
+--------+-----------------+
|     0.0|713.2425021068717|
|     0.2|577.6195582954396|
|    0.05|692.1749257840728|
|     0.1|644.7384521677019|
+--------+-----------------+



In [16]:
## 3. Find the 5 most recent orders (orderBy on order_date)
most_recent = (
    orders
    .orderBy(F.col("order_date").desc())
)

most_recent.show(5)

+--------+-----------+----------+--------+--------+-----------+----------+
|order_id|customer_id|product_id|quantity|discount|order_total|order_date|
+--------+-----------+----------+--------+--------+-----------+----------+
|    3867|        468|         8|       4|     0.0|     359.96|2025-06-19|
|    8703|        717|         9|       4|     0.0|      138.0|2025-06-19|
|    4721|        659|         5|       5|    0.05|    1424.95|2025-06-19|
|     640|        295|         1|       4|     0.1|    3239.96|2025-06-19|
|    5742|       1640|         2|       1|     0.0|      149.5|2025-06-19|
+--------+-----------+----------+--------+--------+-----------+----------+
only showing top 5 rows


In [13]:
## 4. Some orders reference customers that don't exist in customers.csv — how many?
## (Hint: join with how="left_anti".)
ghost_customers = (
    orders.join(customers, on="customer_id", how="left_anti")
)

print(ghost_customers.count())

# ghost_customers2 = (
#     customers.join(orders, on="customer_id", how="left_anti")
# )

# print(ghost_customers.count())


980
980


In [19]:
spark.stop()